In [1]:
# MODEL ARCHITECTURE
"""
Hybrid Recommender System
=======================================================================
Architecture:
  SBERT (all-MiniLM-L6-v2)  → dense content embeddings + FAISS retrieval
  vote_average proxy        → sentiment signal
  UserTransformer           → sequential user behaviour encoding
  NCF (BPR neg-sampling)    → collaborative signal
  Deep Neural Fusion (Keras) → fuses all signals per candidate
  LightGBM LambdaRank        → final listwise ranking
"""

'\nHybrid Recommender System\n=======================================================================\nArchitecture:\n  SBERT (all-MiniLM-L6-v2)  → dense content embeddings + FAISS retrieval\n  vote_average proxy        → sentiment signal\n  UserTransformer           → sequential user behaviour encoding\n  NCF (BPR neg-sampling)    → collaborative signal\n  Deep Neural Fusion (Keras) → fuses all signals per candidate\n  LightGBM LambdaRank        → final listwise ranking\n'

In [2]:
# ── Mount Google Drive for Storing Model ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [3]:
# ── Install ───────────────────────────────────────────────────────────────────
!pip install torch faiss-cpu sentence-transformers lightgbm tensorflow transformers joblib tqdm -q

# ── Imports ───────────────────────────────────────────────────────────────────
import os
import ast
import time
import warnings
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.7 MB/s eta 0:00:00


In [4]:
warnings.filterwarnings("ignore")
logging.basicConfig(
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    level=logging.INFO,
)
log = logging.getLogger("HybridRec")

# ── Configurations ───────────────────────────────────────
CFG = dict(
    movies_path="/content/tmdb_5000_movies.csv",
    credits_path="/content/tmdb_5000_credits.csv",
    ratings_path="/content/ratings.csv",
    links_path="/content/links.csv",
    reviews_path="/content/tmdb_reviews.csv",
    cache_dir="/content/drive/MyDrive/hybrid_rec_cache2",
    ratings_sample=8_000_000,
    test_size=0.2,
    min_ratings_user=20,
    neg_ratio=4,
    sbert_model="all-MiniLM-L6-v2",
    faiss_n_candidates=1000,
    # Transformer
    embed_dim=64,
    proj_dim=384,
    n_heads=4,
    n_layers=2,
    max_seq_len=30,
    transformer_epochs=3,
    transformer_lr=1e-3,
    transformer_batch=128,
    grad_clip=1.0,
    # NCF
    ncf_embed_dim=32,
    ncf_epochs=5,
    ncf_lr=1e-3,
    ncf_batch=2048,
    # Deep fusion
    deep_epochs=5,
    deep_batch=2048,
    # Ranker
    ranker_n_est=200,
    ranker_leaves=63,
    ranker_lr=0.05,
    # Eval
    eval_k=10,
    eval_users=2000,
    device="cuda" if torch.cuda.is_available() else "cpu",
    force_retrain=False,
)

CACHE = Path(CFG["cache_dir"])
os.makedirs(CFG["cache_dir"], exist_ok=True)

print(f"✅ All models will be saved permanently in: {CFG['cache_dir']}")

✅ All models will be saved permanently in: /content/drive/MyDrive/hybrid_rec_cache2


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# GPU CHECK
# ══════════════════════════════════════════════════════════════════════════════
def GPUcheck() -> None:
    print(">>> Checking GPU availability …")
    if torch.cuda.is_available():
        print(f"PyTorch CUDA: {torch.cuda.get_device_name(0)}")
        try:
            _ = torch.ones(2, 2).cuda()
            print("PyTorch: tensor allocated on GPU ✓")
        except Exception as exc:
            log.error(f"PyTorch GPU allocation failed: {exc}")
    else:
        log.warning("PyTorch: CUDA unavailable — running on CPU.")

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"TensorFlow: {len(gpus)} GPU(s) found")
        try:
            with tf.device("/GPU:0"):
                _ = tf.ones([2, 2])
            print("TensorFlow: tensor allocated on GPU ✓")
        except Exception as exc:
            log.error(f"TensorFlow GPU allocation failed: {exc}")
    else:
        log.warning("TensorFlow: no GPUs found — running on CPU.")

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — DATA
# ══════════════════════════════════════════════════════════════════════════════
def _parse_json(x: str, key: str = "name") -> str:
    try:
        return " ".join(i[key] for i in ast.literal_eval(x) if key in i)
    except Exception:
        return ""


def load_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    print("Loading raw data …")
    movies  = pd.read_csv(CFG["movies_path"])
    credits = pd.read_csv(CFG["credits_path"])
    ratings = pd.read_csv(CFG["ratings_path"])
    links   = pd.read_csv(CFG["links_path"])

    if "title" in credits.columns:
        credits = credits.drop(columns=["title"])

    movies = movies.merge(credits, left_on="id", right_on="movie_id", how="left")
    for col in ["genres", "keywords", "cast"]:
        movies[col] = movies[col].apply(_parse_json)
    movies["overview"]     = movies["overview"].fillna("")
    movies["soup"]         = (movies["overview"] + " " + movies["genres"] + " " +
                               movies["keywords"] + " " + movies["cast"])
    movies["vote_average"] = pd.to_numeric(
        movies.get("vote_average", 0), errors="coerce"
    ).fillna(0)

    reviews = None
    rp = CFG.get("reviews_path", "")
    if rp and os.path.exists(rp):
        reviews = pd.read_csv(rp, on_bad_lines="skip")
        print(f"  Reviews loaded: {len(reviews):,} rows")
    else:
        print("  Reviews file not found — using vote_average proxy for sentiment")

    print(f"  Movies: {len(movies):,} | Raw ratings: {len(ratings):,}")
    return movies, ratings, links, reviews


def prepare_data(
    movies: pd.DataFrame,
    ratings: pd.DataFrame,
    links: pd.DataFrame,
) -> Tuple:
    print("Preparing mappings and train/test split …")

    movies = movies.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)
    print(f"  After deduplication: {len(movies):,} unique movies")

    links = links.dropna(subset=["tmdbId"]).copy()
    links["tmdbId"]  = links["tmdbId"].astype(int)
    links["movieId"] = links["movieId"].astype(int)
    valid_mids = set(links["movieId"])

    ratings = ratings[ratings["movieId"].isin(valid_mids)].copy()
    ratings = ratings[(ratings["rating"] >= 0.5) & (ratings["rating"] <= 5.0)]

    n       = min(CFG["ratings_sample"], len(ratings))
    ratings = ratings.sample(n, random_state=42)

    train_r, test_r = train_test_split(ratings, test_size=CFG["test_size"], random_state=42)

    uc         = train_r["userId"].value_counts()
    keep_users = uc[uc >= CFG["min_ratings_user"]].index
    train_r    = train_r[train_r["userId"].isin(keep_users)]

    all_users  = ratings["userId"].unique()
    all_movies = ratings["movieId"].unique()
    user2idx   = {u: i for i, u in enumerate(all_users)}
    movie2idx  = {m: i for i, m in enumerate(all_movies)}

    tmdb_to_movie: Dict[int, int] = links.set_index("tmdbId")["movieId"].to_dict()
    movie_to_tmdb: Dict[int, int] = {v: k for k, v in tmdb_to_movie.items()}

    movie_meta = movies.set_index("id")[["popularity", "vote_average"]].to_dict("index")

    print(f"  Train: {len(train_r):,}  Test: {len(test_r):,}")
    print(f"  Users: {len(user2idx):,}  Items: {len(movie2idx):,}")
    return (train_r, test_r, user2idx, movie2idx,
            tmdb_to_movie, movie_to_tmdb, movie_meta)

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — CONTENT MODEL (SBERT + FAISS)
# ══════════════════════════════════════════════════════════════════════════════
class ContentModel:
    def __init__(self, movies: pd.DataFrame) -> None:
        self.movies     = movies.reset_index(drop=True)
        self.movies     = self.movies.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)
        self.sbert      = SentenceTransformer(CFG["sbert_model"])
        self.embeddings: Optional[np.ndarray] = None
        self.index:      Optional[faiss.Index] = None
        self.tmdb_to_idx: Dict[int, int] = {
            int(row["id"]): i for i, row in self.movies.iterrows()
        }

    def train(self) -> None:
        emb_path = CACHE / "sbert_embeddings.pkl"
        if emb_path.exists() and not CFG["force_retrain"]:
            print("  Loading cached SBERT embeddings …")
            self.embeddings = joblib.load(emb_path)
        else:
            print("  Encoding movies with SBERT …")
            self.embeddings = self.sbert.encode(
                self.movies["soup"].tolist(),
                batch_size=64,
                show_progress_bar=True,
                normalize_embeddings=True,
            ).astype(np.float32)
            joblib.dump(self.embeddings, emb_path)
            print(f"  Embeddings cached → {emb_path}")

        dim        = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.embeddings)
        print(f"  FAISS index: {self.index.ntotal} vectors, dim={dim}")

    def get_embedding(self, tmdb_id: int) -> np.ndarray:
        idx = self.tmdb_to_idx.get(int(tmdb_id))
        if idx is None:
            return np.zeros(self.embeddings.shape[1], dtype=np.float32)
        return self.embeddings[idx]

    def retrieve_candidates(self, query_vec: np.ndarray, k: int = None) -> List[int]:
        k = k or CFG["faiss_n_candidates"]
        q = query_vec.astype(np.float32).reshape(1, -1)
        norm = np.linalg.norm(q)
        if norm > 1e-9:
            q /= norm
        _, indices = self.index.search(q, k)
        return [int(self.movies.iloc[i]["id"]) for i in indices[0] if i >= 0]


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — SENTIMENT MODEL
# ══════════════════════════════════════════════════════════════════════════════
class SentimentModel:
    def build(
        self,
        movies: pd.DataFrame,
        reviews: Optional[pd.DataFrame],
    ) -> Dict[int, float]:
        print("Building sentiment features …")

        va   = movies[["id", "vote_average"]].dropna()
        va   = va[va["vote_average"] > 0]
        vmin = va["vote_average"].min()
        vmax = va["vote_average"].max()
        feats: Dict[int, float] = {
            int(row["id"]): 2 * (row["vote_average"] - vmin) / (vmax - vmin + 1e-9) - 1
            for _, row in va.iterrows()
        }

        if (reviews is not None
                and "content"  in reviews.columns
                and "tmdbId"   in reviews.columns):
            print("  Running DistilBERT sentiment on reviews …")
            pipe     = pipeline(
                "sentiment-analysis",
                model="distilbert-base-uncased-finetuned-sst-2-english",
                top_k=None, device=-1, batch_size=32,
            )
            rev_dict = reviews.groupby("tmdbId")["content"].apply(list).to_dict()
            for tmdb_id, texts in tqdm(rev_dict.items(), desc="  Sentiment"):
                texts = [str(t)[:512] for t in texts[:20]]
                try:
                    results = pipe(texts, truncation=True, max_length=512)
                    vals = []
                    for res in results:
                        top = max(res, key=lambda x: x["score"])
                        vals.append(
                            top["score"] if top["label"] == "POSITIVE" else -top["score"]
                        )
                    feats[int(tmdb_id)] = float(np.mean(vals))
                except Exception:
                    pass

        print(f"  Sentiment map: {len(feats):,} entries")
        return feats

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 4 — USER TRANSFORMER
# ══════════════════════════════════════════════════════════════════════════════

class UserTransformer(nn.Module):
    def __init__(self, n_items: int) -> None:
        super().__init__()
        ed  = CFG["embed_dim"]
        pd_ = CFG["proj_dim"]
        self.item_emb = nn.Embedding(n_items + 1, ed, padding_idx=0)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=ed, nhead=CFG["n_heads"], batch_first=True,
            dim_feedforward=ed * 4, dropout=0.1, activation="relu",
        )
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=CFG["n_layers"])
        self.user_proj  = nn.Linear(ed, pd_)
        self.target_proj = nn.Linear(ed, pd_)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        B, T = seq.shape
        if T == 0:
            zero_emb = torch.zeros(B, CFG["embed_dim"], device=seq.device)
            return self.user_proj(zero_emb)

        x    = self.item_emb(seq)
        mask = (seq == 0)

        if mask.all():
            zero_emb = torch.zeros(B, CFG["embed_dim"], device=seq.device, dtype=x.dtype)
            return self.user_proj(zero_emb)

        out = self.encoder(x, src_key_padding_mask=mask)

        non_pad  = (~mask).float().unsqueeze(-1)
        sum_out  = (out * non_pad).sum(dim=1)
        count    = non_pad.sum(dim=1).clamp(min=1)
        mean_out = sum_out / count

        return self.user_proj(mean_out)

    def target_embed(self, items: torch.Tensor) -> torch.Tensor:
        return self.target_proj(self.item_emb(items))


class SeqDataset(TorchDataset):
    def __init__(self, sequences: Dict, max_len: int) -> None:
        self.data    = [(uid, seq) for uid, seq in sequences.items() if len(seq) >= 2]
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        uid, seq = self.data[idx]
        inp      = seq[:-1][-self.max_len:]
        target   = seq[-1]
        pad      = [0] * (self.max_len - len(inp))
        return torch.LongTensor(pad + inp), torch.LongTensor([target])


def build_user_sequences(ratings: pd.DataFrame, movie2idx: Dict) -> Dict:
    print("Building user sequences …")
    seqs: Dict = {}
    for uid, grp in tqdm(ratings.groupby("userId"), desc="  Sequences"):
        seq = [
            movie2idx[m] + 1
            for m in grp.sort_values("timestamp")["movieId"]
            if m in movie2idx
        ]
        if seq:
            seqs[uid] = seq[-CFG["max_seq_len"]:]
    print(f"  {len(seqs):,} user sequences built")
    return seqs


def _pad_seq(seq: List[int], max_len: int) -> List[int]:
    seq = seq[-max_len:]
    return [0] * (max_len - len(seq)) + seq


def train_user_model(sequences: Dict, n_items: int) -> UserTransformer:
    cache_path = CACHE / "user_transformer.pt"   # name no longer embeds n_items
    device     = CFG["device"]

    if cache_path.exists() and not CFG["force_retrain"]:
        print("  Loading cached UserTransformer …")
        # Load the checkpoint dict which contains both dims and weights
        ckpt   = torch.load(cache_path, map_location=device)
        # Use the saved n_items, NOT the current data's n_items
        saved_n_items = ckpt["n_items"]
        model  = UserTransformer(saved_n_items).to(device)
        model.load_state_dict(ckpt["state_dict"])
        model.eval()
        print(f"  Loaded with n_items={saved_n_items} (current data n_items={n_items})")
        return model

    print("Training UserTransformer …")
    model  = UserTransformer(n_items).to(device)
    optim  = torch.optim.AdamW(model.parameters(), lr=CFG["transformer_lr"], weight_decay=1e-4)
    ds     = SeqDataset(sequences, CFG["max_seq_len"])
    loader = DataLoader(ds, batch_size=CFG["transformer_batch"], shuffle=True, num_workers=0)

    for epoch in range(CFG["transformer_epochs"]):
        model.train()
        total_loss, steps = 0.0, 0
        pbar = tqdm(loader, desc=f"  Epoch {epoch+1}/{CFG['transformer_epochs']}")
        for inp, target in pbar:
            inp, target = inp.to(device), target.squeeze(1).to(device)
            user_vec = model(inp)
            tgt_vec  = model.target_embed(target)
            loss     = 1.0 - F.cosine_similarity(user_vec, tgt_vec).mean()
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optim.step()
            total_loss += loss.item()
            steps      += 1
            pbar.set_postfix(loss=f"{total_loss/steps:.4f}")
        print(f"  Epoch {epoch+1} avg loss: {total_loss/steps:.4f}")

    # Save as dict containing dims + weights
    torch.save({"n_items": n_items, "state_dict": model.state_dict()}, cache_path)
    print(f"  UserTransformer cached → {cache_path}")
    model.eval()
    return model

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 5 — NCF WITH BPR NEGATIVE SAMPLING
# ══════════════════════════════════════════════════════════════════════════════

class NCF(nn.Module):
    def __init__(self, n_users: int, n_items: int) -> None:
        super().__init__()
        ed = CFG["ncf_embed_dim"]
        self.user_emb = nn.Embedding(n_users, ed)
        self.item_emb = nn.Embedding(n_items + 1, ed, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(ed * 2, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),     nn.ReLU(),
            nn.Linear(64, 1),
        )
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def score(self, u: torch.Tensor, i: torch.Tensor) -> torch.Tensor:
        return self.mlp(torch.cat([self.user_emb(u), self.item_emb(i)], dim=1)).squeeze(1)

    def bpr_loss(self, u: torch.Tensor, pos_i: torch.Tensor, neg_i: torch.Tensor) -> torch.Tensor:
        return -F.logsigmoid(self.score(u, pos_i) - self.score(u, neg_i)).mean()


class NCFDataset(TorchDataset):
    def __init__(
        self,
        df: pd.DataFrame,
        user2idx: Dict,
        movie2idx: Dict,
        neg_ratio: int,
        all_movie_idxs: np.ndarray,
    ) -> None:
        self.records        = df[["userId", "movieId"]].values
        self.u2i            = user2idx
        self.m2i            = movie2idx
        self.neg_ratio      = neg_ratio
        self.all_movie_idxs = all_movie_idxs

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        uid, mid = self.records[idx]
        uidx     = self.u2i[uid]
        midx     = self.m2i[mid] + 1
        neg_idx  = int(np.random.choice(self.all_movie_idxs)) + 1
        return (
            torch.LongTensor([uidx]),
            torch.LongTensor([midx]),
            torch.LongTensor([neg_idx]),
        )

def train_ncf(train_df: pd.DataFrame, user2idx: Dict, movie2idx: Dict) -> NCF:
    cache_path = CACHE / "ncf_v2.pt"
    device     = CFG["device"]
    n_users    = len(user2idx)
    n_items    = len(movie2idx)

    if cache_path.exists() and not CFG["force_retrain"]:
        print("  Loading cached NCF …")
        ckpt = torch.load(cache_path, map_location=device)

        # ── Backward-compat: old checkpoints were bare state_dicts ──────────
        if isinstance(ckpt, dict) and "n_users" in ckpt:
            saved_n_users = ckpt["n_users"]
            saved_n_items = ckpt["n_items"]
            state_dict    = ckpt["state_dict"]
        else:
            # Old-format checkpoint — dimensions must match; if they don't,
            # log a warning and fall through to retraining instead of crashing.
            log.warning(
                "  Old-format NCF checkpoint detected (no dimension metadata). "
                "Checking if dimensions match current data …"
            )
            saved_n_users = ckpt["user_emb.weight"].shape[0]
            saved_n_items = ckpt["item_emb.weight"].shape[0] - 1  # accounts for padding_idx
            state_dict    = ckpt

            if saved_n_users != n_users or saved_n_items != n_items:
                log.warning(
                    f"  Dimension mismatch: checkpoint=({saved_n_users}, {saved_n_items}) "
                    f"vs current=({n_users}, {n_items}). Retraining NCF …"
                )
                # Delete stale cache and fall through to training block below
                os.remove(cache_path)
                return train_ncf(train_df, user2idx, movie2idx)

        model = NCF(saved_n_users, saved_n_items).to(device)
        model.load_state_dict(state_dict)
        model.eval()
        print(
            f"  NCF loaded: n_users={saved_n_users}, n_items={saved_n_items} "
            f"(current data: {n_users}, {n_items})"
        )
        return model

    print("Training NCF (BPR) …")
    model          = NCF(n_users, n_items).to(device)
    optim          = torch.optim.Adam(model.parameters(), lr=CFG["ncf_lr"])
    all_movie_idxs = np.array(list(movie2idx.values()))
    ds             = NCFDataset(train_df, user2idx, movie2idx, CFG["neg_ratio"], all_movie_idxs)
    loader         = DataLoader(ds, batch_size=CFG["ncf_batch"], shuffle=True, num_workers=0)

    for epoch in range(CFG["ncf_epochs"]):
        model.train()
        total_loss, steps = 0.0, 0
        pbar = tqdm(loader, desc=f"  NCF Epoch {epoch+1}/{CFG['ncf_epochs']}")
        for u, pos, neg in pbar:
            u, pos, neg = u.squeeze(1).to(device), pos.squeeze(1).to(device), neg.squeeze(1).to(device)
            loss = model.bpr_loss(u, pos, neg)
            optim.zero_grad()
            loss.backward()
            optim.step()
            total_loss += loss.item()
            steps      += 1
            pbar.set_postfix(loss=f"{total_loss/steps:.4f}")
        print(f"  NCF Epoch {epoch+1} loss: {total_loss/steps:.4f}")

    # Save as dict containing dims + weights
    torch.save(
        {"n_users": n_users, "n_items": n_items, "state_dict": model.state_dict()},
        cache_path,
    )
    print(f"  NCF cached → {cache_path}")
    model.eval()
    return model

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 6 — DEEP FUSION MODEL
# ══════════════════════════════════════════════════════════════════════════════

def build_deep_model(n_users: int, n_items: int, text_dim: int) -> Model:
    u_in    = Input(shape=(1,),        name="user")
    i_in    = Input(shape=(1,),        name="item")
    text_in = Input(shape=(text_dim,), name="text")
    sent_in = Input(shape=(1,),        name="sent")

    u = Flatten()(Embedding(n_users,     32, name="user_emb")(u_in))
    i = Flatten()(Embedding(n_items + 1, 32, name="item_emb")(i_in))

    x   = Concatenate()([u, i, text_in, sent_in])
    x   = Dense(256, activation="relu")(x)
    x   = Dropout(0.3)(x)
    x   = Dense(128, activation="relu")(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64,  activation="relu")(x)
    out = Dense(1)(x)

    model = Model([u_in, i_in, text_in, sent_in], out)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def train_deep_model(
    train_df:       pd.DataFrame,
    user2idx:       Dict,
    movie2idx:      Dict,
    content:        ContentModel,
    sent_feats:     Dict[int, float],
    movie_to_tmdb:  Dict[int, int],
) -> Model:
    cache_path = CACHE / "deep_model.keras"
    n_users    = len(user2idx)
    n_items    = len(movie2idx)
    text_dim   = content.embeddings.shape[1]

    if cache_path.exists() and not CFG["force_retrain"]:
        print("  Loading cached Deep model …")
        return tf.keras.models.load_model(str(cache_path))

    print("Training Deep Fusion model …")
    model     = build_deep_model(n_users, n_items, text_dim)
    sample    = train_df.sample(min(250_000, len(train_df)), random_state=42)
    all_mids  = list(movie2idx.keys())

    rows_u, rows_i, rows_t, rows_s, rows_y = [], [], [], [], []

    print("  Building deep training matrix …")
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="  Deep features"):
        uid = row["userId"]
        mid = int(row["movieId"])
        if uid not in user2idx or mid not in movie2idx:
            continue
        tmdb = movie_to_tmdb.get(mid, 0)
        uidx = user2idx[uid]
        midx = movie2idx[mid] + 1

        rows_u.append(uidx)
        rows_i.append(midx)
        rows_t.append(content.get_embedding(tmdb))
        rows_s.append(sent_feats.get(tmdb, 0.0))
        rows_y.append(row["rating"] / 5.0)

        for _ in range(CFG["neg_ratio"]):
            neg_mid  = int(np.random.choice(all_mids))
            neg_tmdb = movie_to_tmdb.get(neg_mid, 0)
            rows_u.append(uidx)
            rows_i.append(movie2idx.get(neg_mid, 0) + 1)
            rows_t.append(content.get_embedding(neg_tmdb))
            rows_s.append(sent_feats.get(neg_tmdb, 0.0))
            rows_y.append(0.0)

    X_u = np.array(rows_u, dtype=np.int32).reshape(-1, 1)
    X_i = np.array(rows_i, dtype=np.int32).reshape(-1, 1)
    X_t = np.array(rows_t, dtype=np.float32)
    X_s = np.array(rows_s, dtype=np.float32).reshape(-1, 1)
    y   = np.array(rows_y, dtype=np.float32)

    print(f"  Deep training set: {len(y):,} samples")
    cb = EarlyStopping(patience=2, restore_best_weights=True, verbose=0)
    model.fit(
        [X_u, X_i, X_t, X_s], y,
        epochs=CFG["deep_epochs"],
        batch_size=CFG["deep_batch"],
        validation_split=0.1,
        callbacks=[cb],
        verbose=1,
    )
    model.save(str(cache_path))
    print(f"  Deep model cached → {cache_path}")
    return model

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 7 — FEATURE BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def _tmdb_to_internal_idx(
    candidate_tmdb_ids: List[int],
    tmdb_to_movie:      Dict[int, int],
    movie2idx:          Dict[int, int],
) -> np.ndarray:
    idxs = []
    for tmdb in candidate_tmdb_ids:
        ml_id    = tmdb_to_movie.get(int(tmdb), -1)
        internal = movie2idx.get(ml_id, -1)
        idxs.append(max(internal + 1, 0))
    return np.array(idxs, dtype=np.int64)


def build_features(
    uid:                int,
    uidx:               int,
    candidate_tmdb_ids: List[int],
    user_model:         UserTransformer,
    ncf:                NCF,
    deep:               Model,
    content:            ContentModel,
    sent_feats:         Dict[int, float],
    movie_meta:         Dict,
    tmdb_to_movie:      Dict[int, int],
    movie2idx:          Dict[int, int],
    user_sequences:     Dict,
    device:             str,
) -> np.ndarray:
    N = len(candidate_tmdb_ids)

    raw_seq = user_sequences.get(uid, [0])
    seq     = _pad_seq(raw_seq, CFG["max_seq_len"])
    seq_t   = torch.LongTensor(seq).unsqueeze(0).to(device)
    with torch.no_grad():
        user_vec = user_model(seq_t).cpu().numpy()[0]
    user_vec /= (np.linalg.norm(user_vec) + 1e-9)

    item_embs   = np.array(
        [content.get_embedding(t) for t in candidate_tmdb_ids], dtype=np.float32
    )
    content_sim = item_embs @ user_vec

    sent_arr = np.array(
        [sent_feats.get(t, 0.0) for t in candidate_tmdb_ids], dtype=np.float32
    )

    pop_arr  = np.array(
        [movie_meta.get(t, {}).get("popularity",    0) / 100
         for t in candidate_tmdb_ids], dtype=np.float32
    )
    vote_arr = np.array(
        [movie_meta.get(t, {}).get("vote_average",  0) / 10
         for t in candidate_tmdb_ids], dtype=np.float32
    )

    movie_idxs = _tmdb_to_internal_idx(candidate_tmdb_ids, tmdb_to_movie, movie2idx)

    # ── Clamp indices to the embedding table sizes actually in each model ────
    ncf_max_user  = ncf.user_emb.weight.shape[0] - 1
    ncf_max_item  = ncf.item_emb.weight.shape[0] - 1
    u_tensor = torch.LongTensor([min(uidx, ncf_max_user)] * N).to(device)
    i_tensor = torch.LongTensor(np.clip(movie_idxs, 0, ncf_max_item)).to(device)

    with torch.no_grad():
        cf_scores = ncf.score(u_tensor, i_tensor).cpu().numpy()

    deep_scores = deep.predict(
        [
            np.array([uidx] * N, dtype=np.int32).reshape(-1, 1),
            movie_idxs.reshape(-1, 1).astype(np.int32),
            item_embs,
            sent_arr.reshape(-1, 1),
        ],
        verbose=0,
        batch_size=512,
    ).flatten()

    X = np.column_stack([cf_scores, deep_scores, content_sim, pop_arr, vote_arr, sent_arr])
    return X.astype(np.float32)

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 8 — LIGHTGBM LAMBDARANK
# ══════════════════════════════════════════════════════════════════════════════

class Ranker:
    FEATURE_NAMES = ["ncf", "deep", "content_sim", "popularity", "vote_avg", "sentiment"]

    def __init__(self) -> None:
        self.model = lgb.LGBMRanker(
            objective="lambdarank",
            n_estimators=CFG["ranker_n_est"],
            num_leaves=CFG["ranker_leaves"],
            learning_rate=CFG["ranker_lr"],
            min_child_samples=5,
            n_jobs=-1,
            verbose=-1,
            label_gain=[0, 1, 3, 7, 15, 31],
        )

    def train(self, X: np.ndarray, y: np.ndarray, group_sizes: List[int]) -> None:
        print(f"  Training LambdaRank on {len(y):,} samples …")
        y_int = np.clip(np.round(y).astype(np.int32), 0, 5)
        self.model.fit(X, y_int, group=group_sizes, feature_name=self.FEATURE_NAMES)
        print("  LambdaRank trained")

        print("\n" + "=" * 40)
        print("EXPLAINABLE AI: FEATURE IMPORTANCE (GAIN)")
        print("=" * 40)
        importance = self.model.feature_importances_
        total_gain = np.sum(importance)
        for name, imp in zip(self.FEATURE_NAMES, importance):
            print(f"  {name:<15}: {imp/total_gain * 100:.1f}% contribution")
        print("=" * 40 + "\n")

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)


def train_ranker(
    train_df:       pd.DataFrame,
    user2idx:       Dict,
    movie2idx:      Dict,
    user_model:     UserTransformer,
    ncf:            NCF,
    deep:           Model,
    content:        ContentModel,
    sent_feats:     Dict[int, float],
    movie_meta:     Dict,
    tmdb_to_movie:  Dict,
    movie_to_tmdb:  Dict,
    user_sequences: Dict,
) -> Ranker:
    cache_path = CACHE / "ranker_v2.pkl"

    if cache_path.exists() and not CFG["force_retrain"]:
        print("  Loading cached Ranker …")
        return joblib.load(cache_path)

    print("Building ranker training features …")
    device  = CFG["device"]
    ranker  = Ranker()
    sample  = train_df.sample(min(100_000, len(train_df)), random_state=42)

    X_parts:     List[np.ndarray] = []
    y_parts:     List[float]      = []
    group_sizes: List[int]        = []

    for uid, grp_df in tqdm(sample.groupby("userId"), desc="  Ranker features"):
        if uid not in user2idx or len(grp_df) < 2:
            continue
        uidx    = user2idx[uid]
        tmdb_ids = [movie_to_tmdb.get(int(m), 0) for m in grp_df["movieId"]]
        valid    = [(t, float(r)) for t, r in zip(tmdb_ids, grp_df["rating"]) if t != 0]
        if not valid:
            continue
        t_ids, y_vals = zip(*valid)

        feats = build_features(
            uid, uidx, list(t_ids), user_model, ncf, deep,
            content, sent_feats, movie_meta, tmdb_to_movie,
            movie2idx, user_sequences, device,
        )
        X_parts.append(feats)
        y_parts.extend(y_vals)
        group_sizes.append(len(t_ids))

    if not X_parts:
        raise RuntimeError("No ranker training samples were built — check data pipeline.")

    X_all = np.vstack(X_parts)
    y_all = np.array(y_parts)
    ranker.train(X_all, y_all, group_sizes)
    joblib.dump(ranker, cache_path)
    print(f"  Ranker cached → {cache_path}")
    return ranker


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS FOR FUTURE INFERENCE (PERMANENT ON DRIVE)
# ══════════════════════════════════════════════════════════════════════════════
def save_inference_artifacts(
    user2idx, movie2idx, tmdb_to_movie, movie_to_tmdb,
    movie_meta, user_sequences, movies, sent_feats, content
):
    artifacts = {
        "user2idx": user2idx,
        "movie2idx": movie2idx,
        "tmdb_to_movie": tmdb_to_movie,
        "movie_to_tmdb": movie_to_tmdb,
        "movie_meta": movie_meta,
        "user_sequences": user_sequences,
        "sent_feats": sent_feats,
        "movies_df": movies,
    }
    path = CACHE / "inference_artifacts.pkl"
    joblib.dump(artifacts, path)
    print(f"✅ All inference artifacts saved permanently: {path}")

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 9 — EVALUATION
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(
    test_df:        pd.DataFrame,
    user2idx:       Dict,
    movie2idx:      Dict,
    user_model:     UserTransformer,
    ncf:            NCF,
    deep:           Model,
    content:        ContentModel,
    sent_feats:     Dict[int, float],
    movie_meta:     Dict,
    tmdb_to_movie:  Dict,
    movie_to_tmdb:  Dict,
    user_sequences: Dict,
    ranker:         Ranker,
    k:              int = None,
) -> Dict[str, float]:
    k      = k or CFG["eval_k"]
    device = CFG["device"]
    print(f"\n>>> Academic Evaluation: NDCG@{k}, HR@{k} (1-vs-99 Protocol) …")

    pos_test = test_df[test_df["rating"] >= 4.0]
    uc       = pos_test["userId"].value_counts()
    eligible = uc[uc >= 1].index.tolist()

    rng      = np.random.default_rng(42)
    sample_u = rng.choice(eligible, size=min(CFG["eval_users"], len(eligible)), replace=False)

    all_tmdbs = list(tmdb_to_movie.keys())

    ndcg_list: List[float] = []
    hr_list:   List[int]   = []

    for uid in tqdm(sample_u, desc=f"  Eval HR@{k}"):
        if uid not in user2idx:
            continue
        uidx    = user2idx[uid]
        uid_pos = pos_test[pos_test["userId"] == uid]

        true_row  = uid_pos.sample(1, random_state=int(uid)).iloc[0]
        true_mid  = int(true_row["movieId"])
        true_tmdb = movie_to_tmdb.get(true_mid, 0)
        if not true_tmdb:
            continue

        user_history_mids = set(user_sequences.get(uid, []))
        neg_tmdbs = []
        while len(neg_tmdbs) < 99:
            cand = int(rng.choice(all_tmdbs))
            cand_mid = tmdb_to_movie.get(cand, -1)
            if cand != true_tmdb and cand_mid not in user_history_mids:
                neg_tmdbs.append(cand)

        test_items = [true_tmdb] + neg_tmdbs

        raw_seq = user_sequences.get(uid, [0])
        seq     = _pad_seq(raw_seq, CFG["max_seq_len"])
        seq_t   = torch.LongTensor(seq).unsqueeze(0).to(device)

        feats = build_features(
            uid, uidx, test_items, user_model, ncf, deep,
            content, sent_feats, movie_meta, tmdb_to_movie,
            movie2idx, user_sequences, device,
        )

        scores   = ranker.predict(feats)
        top_idx  = np.argsort(scores)[::-1][:k]
        top_tmdbs = [test_items[i] for i in top_idx]

        hr_list.append(int(true_tmdb in top_tmdbs))

        rel_true = np.zeros(len(test_items))
        rel_true[0] = 1.0
        ndcg_list.append(ndcg_score([rel_true], [scores], k=k))

    results = {
        f"HR@{k}":   float(np.mean(hr_list))   if hr_list   else 0.0,
        f"NDCG@{k}": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
        "n_eval":    len(hr_list),
    }

    print("\n" + "-" * 40)
    print(f"ACADEMIC METRICS (1-vs-99 Protocol, N={results['n_eval']})")
    print("-" * 40)
    for key, val in results.items():
        if key != "n_eval":
            print(f"  {key:<12} = {val:.4f}")
    print("-" * 40)

    return results


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 10 — RECOMMEND (WITH COLD-START HANDLING)
# ══════════════════════════════════════════════════════════════════════════════
def recommend(
    uid:            int,
    top_k:          int              = 10,
    user2idx:       Dict             = None,
    movie2idx:      Dict             = None,
    user_model:     UserTransformer  = None,
    ncf:            NCF              = None,
    deep:           Model            = None,
    content:        ContentModel     = None,
    sent_feats:     Dict             = None,
    movie_meta:     Dict             = None,
    tmdb_to_movie:  Dict             = None,
    movie_to_tmdb:  Dict             = None,
    user_sequences: Dict             = None,
    ranker:         Ranker           = None,
    movies:         pd.DataFrame     = None,
) -> List[Dict]:
    """Call this directly anytime after first training. Works for new users too!"""
    device = CFG["device"]

    # ── COLD-START HANDLING (brand-new user with no history) ─────────────────
    if uid not in user2idx:
        print(f"Cold-start user {uid} → using popularity + sentiment fallback")
        global_scores = []
        for _, row in movies.iterrows():
            tmdb = int(row["id"])
            pop  = row.get("popularity", 0) / 100
            vote = row.get("vote_average", 0) / 10
            sent = sent_feats.get(tmdb, 0.0)
            score = 0.4 * pop + 0.4 * vote + 0.2 * sent
            global_scores.append((tmdb, score, row.get("title", f"TMDB_{tmdb}")))

        global_scores.sort(key=lambda x: x[1], reverse=True)
        return [
            {"tmdb_id": t[0], "title": t[2], "score": float(t[1])}
            for t in global_scores[:top_k]
        ]

    # ── NORMAL RECOMMENDATION (known user) ───────────────────────────────────
    uidx    = user2idx[uid]
    raw_seq = user_sequences.get(uid, [0])
    seq     = _pad_seq(raw_seq, CFG["max_seq_len"])
    seq_t   = torch.LongTensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        user_vec = user_model(seq_t).cpu().numpy()[0]

    cand_tmdb = content.retrieve_candidates(user_vec, k=CFG["faiss_n_candidates"])
    feats     = build_features(
        uid, uidx, cand_tmdb, user_model, ncf, deep,
        content, sent_feats, movie_meta, tmdb_to_movie,
        movie2idx, user_sequences, device,
    )
    scores    = ranker.predict(feats)
    top_idx   = np.argsort(scores)[::-1][:top_k]

    title_col = "title" if "title" in movies.columns else "title_x" if "title_x" in movies.columns else "original_title"
    title_map = movies.set_index("id")[title_col].to_dict() if title_col in movies.columns else {}

    return [
        {
            "tmdb_id": cand_tmdb[i],
            "title":   title_map.get(cand_tmdb[i], f"TMDB_{cand_tmdb[i]}"),
            "score":   float(scores[i]),
        }
        for i in top_idx
    ]


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    GPUcheck()
    t_start = time.time()

    # ── Data ──────────────────────────────────────────────────────────────────
    movies, ratings_raw, links, reviews = load_data()
    (train_r, test_r, user2idx, movie2idx,
     tmdb_to_movie, movie_to_tmdb, movie_meta) = prepare_data(movies, ratings_raw, links)

    # ── Models (will load from Drive if already trained) ──────────────────────
    content    = ContentModel(movies)
    content.train()

    sent_feats = SentimentModel().build(movies, reviews)

    user_sequences = build_user_sequences(train_r, movie2idx)
    user_model     = train_user_model(user_sequences, len(movie2idx))
    ncf            = train_ncf(train_r, user2idx, movie2idx)
    deep           = train_deep_model(
        train_r, user2idx, movie2idx, content, sent_feats, movie_to_tmdb
    )
    ranker = train_ranker(
        train_r, user2idx, movie2idx, user_model, ncf, deep,
        content, sent_feats, movie_meta, tmdb_to_movie, movie_to_tmdb, user_sequences,
    )

    # ── Save everything permanently for future sessions ───────────────────────
    save_inference_artifacts(
        user2idx, movie2idx, tmdb_to_movie, movie_to_tmdb,
        movie_meta, user_sequences, movies, sent_feats, content
    )

    print(f"\nTotal training time: {(time.time()-t_start)/60:.1f} min")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    metrics = evaluate(
        test_r, user2idx, movie2idx, user_model, ncf, deep,
        content, sent_feats, movie_meta, tmdb_to_movie, movie_to_tmdb,
        user_sequences, ranker,
    )

    # ── Demo recommendations ──────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("RECOMMENDATIONS")
    print("=" * 60)
    demo_uid = list(user2idx.keys())[0]
    recs = recommend(
        demo_uid, top_k=10,
        user2idx=user2idx, movie2idx=movie2idx,
        user_model=user_model, ncf=ncf, deep=deep, content=content,
        sent_feats=sent_feats, movie_meta=movie_meta,
        tmdb_to_movie=tmdb_to_movie, movie_to_tmdb=movie_to_tmdb,
        user_sequences=user_sequences, ranker=ranker, movies=movies,
    )
    print(f"\nTop-10 for user {demo_uid}:")
    for rank, r in enumerate(recs, 1):
        print(f"  {rank:2d}. {r['title']:<45}  score={r['score']:.4f}")

    print("\nDONE! Model is now saved permanently in Google Drive.")

>>> Checking GPU availability …
Loading raw data …
  Reviews loaded: 7,369 rows
  Movies: 4,803 | Raw ratings: 25,000,095
Preparing mappings and train/test split …
  After deduplication: 4,803 unique movies
  Train: 5,544,942  Test: 1,600,000
  Users: 162,536  Items: 45,487


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Deep training set: 1,250,000 samples
Epoch 1/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 59s 101ms/step - loss: 0.0415 - mae: 0.1088 - val_loss: 0.0385 - val_mae: 0.0985
Epoch 2/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 55s 99ms/step - loss: 0.0356 - mae: 0.0943 - val_loss: 0.0372 - val_mae: 0.0988
Epoch 3/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 53s 97ms/step - loss: 0.0311 - mae: 0.0854 - val_loss: 0.0370 - val_mae: 0.0949
Epoch 4/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 56s 102ms/step - loss: 0.0259 - mae: 0.0740 - val_loss: 0.0390 - val_mae: 0.0970
Epoch 5/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 81s 101ms/step - loss: 0.0207 - mae: 0.0625 - val_loss: 0.0411 - val_mae: 0.0938
  Deep model cached → /content/drive/MyDrive/hybrid_rec_cache2/deep_model.keras
Building ranker training features …


  Ranker features: 100%|██████████| 45828/45828 [41:29<00:00, 18.41it/s]


  Training LambdaRank on 77,043 samples …
  LambdaRank trained

EXPLAINABLE AI: FEATURE IMPORTANCE (GAIN)
  ncf            : 23.8% contribution
  deep           : 24.2% contribution
  content_sim    : 14.9% contribution
  popularity     : 14.5% contribution
  vote_avg       : 10.2% contribution
  sentiment      : 12.3% contribution

  Ranker cached → /content/drive/MyDrive/hybrid_rec_cache2/ranker_v2.pkl
✅ All inference artifacts saved permanently: /content/drive/MyDrive/hybrid_rec_cache2/inference_artifacts.pkl

Total training time: 214.4 min

>>> Academic Evaluation: NDCG@10, HR@10 (1-vs-99 Protocol) …


  Eval HR@10: 100%|██████████| 2000/2000 [18:00<00:00,  1.85it/s]



----------------------------------------
ACADEMIC METRICS (1-vs-99 Protocol, N=2000)
----------------------------------------
  HR@10        = 0.8380
  NDCG@10      = 0.6136
----------------------------------------

RECOMMENDATIONS

Top-10 for user 21365:
   1. The Dark Knight                                score=2.6431
   2. The Bridge on the River Kwai                   score=2.4748
   3. Raiders of the Lost Ark                        score=2.3203
   4. Blade Runner                                   score=2.2579
   5. Gladiator                                      score=2.1854
   6. 12 Angry Men                                   score=2.1471
   7. Forrest Gump                                   score=2.1129
   8. The Matrix                                     score=2.0954
   9. Psycho                                         score=2.0742
  10. The Longest Day                                score=1.8717

DONE! Model is now saved permanently in Google Drive.


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD AND USE SAVED MODELS
# ══════════════════════════════════════════════════════════════════════════════
import joblib
import tensorflow as tf
import torch
import pandas as pd
import os
from pathlib import Path

CACHE = Path(CFG["cache_dir"])

# 1. Load inference artifacts
artifacts_path = CACHE / "inference_artifacts.pkl"
if artifacts_path.exists():
    print(f"Loading inference artifacts from: {artifacts_path}")
    artifacts = joblib.load(artifacts_path)

    user2idx        = artifacts["user2idx"]
    movie2idx       = artifacts["movie2idx"]
    tmdb_to_movie   = artifacts["tmdb_to_movie"]
    movie_to_tmdb   = artifacts["movie_to_tmdb"]
    movie_meta      = artifacts["movie_meta"]
    user_sequences  = artifacts["user_sequences"]
    movies_df       = artifacts["movies_df"]
    sent_feats      = artifacts["sent_feats"]

    print("Inference artifacts loaded successfully.")
else:
    raise FileNotFoundError(f"Inference artifacts not found at {artifacts_path}. Please run the training pipeline first.")

# 2. Load UserTransformer model
user_transformer_path = CACHE / "user_transformer.pt"
if user_transformer_path.exists():
    print(f"Loading UserTransformer from: {user_transformer_path}")
    # Need to instantiate with the correct n_items used during training
    ckpt = torch.load(user_transformer_path, map_location=CFG["device"])
    saved_n_items = ckpt["n_items"]
    user_model = UserTransformer(saved_n_items).to(CFG["device"])
    user_model.load_state_dict(ckpt["state_dict"])
    user_model.eval()
    print("UserTransformer loaded successfully.")
else:
    raise FileNotFoundError(f"UserTransformer model not found at {user_transformer_path}. Please run the training pipeline first.")

# 3. Load NCF model
ncf_path = CACHE / "ncf_v2.pt"
if ncf_path.exists():
    print(f"Loading NCF from: {ncf_path}")
    ckpt = torch.load(ncf_path, map_location=CFG["device"])
    # Backward-compatible loading for NCF
    if isinstance(ckpt, dict) and "n_users" in ckpt:
        saved_n_users = ckpt["n_users"]
        saved_n_items = ckpt["n_items"]
        state_dict    = ckpt["state_dict"]
    else:
        saved_n_users = ckpt["user_emb.weight"].shape[0]
        saved_n_items = ckpt["item_emb.weight"].shape[0] - 1
        state_dict    = ckpt

    ncf = NCF(saved_n_users, saved_n_items).to(CFG["device"])
    ncf.load_state_dict(state_dict)
    ncf.eval()
    print("NCF model loaded successfully.")
else:
    raise FileNotFoundError(f"NCF model not found at {ncf_path}. Please run the training pipeline first.")

# 4. Load Deep Fusion model
deep_path = CACHE / "deep_model.keras"
if deep_path.exists():
    print(f"Loading Deep Fusion model from: {deep_path}")
    deep_model = tf.keras.models.load_model(str(deep_path))
    print("Deep Fusion model loaded successfully.")
else:
    raise FileNotFoundError(f"Deep Fusion model not found at {deep_path}. Please run the training pipeline first.")

# 5. Load Ranker model
ranker_path = CACHE / "ranker_v2.pkl"
if ranker_path.exists():
    print(f"Loading Ranker from: {ranker_path}")
    ranker = joblib.load(ranker_path)
    print("Ranker model loaded successfully.")
else:
    raise FileNotFoundError(f"Ranker model not found at {ranker_path}. Please run the training pipeline first.")

# 6. Re-initialize ContentModel (only its sbert and faiss index)
content_model = ContentModel(movies_df)
content_model.train()
print("ContentModel (SBERT + FAISS) re-initialized.")


Loading inference artifacts from: /content/drive/MyDrive/hybrid_rec_cache2/inference_artifacts.pkl
Inference artifacts loaded successfully.
Loading UserTransformer from: /content/drive/MyDrive/hybrid_rec_cache2/user_transformer.pt
UserTransformer loaded successfully.
Loading NCF from: /content/drive/MyDrive/hybrid_rec_cache2/ncf_v2.pt
NCF model loaded successfully.
Loading Deep Fusion model from: /content/drive/MyDrive/hybrid_rec_cache2/deep_model.keras
Deep Fusion model loaded successfully.
Loading Ranker from: /content/drive/MyDrive/hybrid_rec_cache2/ranker_v2.pkl
Ranker model loaded successfully.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loading cached SBERT embeddings …
  FAISS index: 4803 vectors, dim=384
ContentModel (SBERT + FAISS) re-initialized.


In [26]:
demo_uid_=int(input('Enter user id: '))

# Get recommendations using the loaded models and artifacts
loaded_recs = recommend(
    uid=demo_uid_,
    top_k=10,
    user2idx=user2idx,
    movie2idx=movie2idx,
    user_model=user_model,
    ncf=ncf,
    deep=deep_model,
    content=content_model,
    sent_feats=sent_feats,
    movie_meta=movie_meta,
    tmdb_to_movie=tmdb_to_movie,
    movie_to_tmdb=movie_to_tmdb,
    user_sequences=user_sequences,
    ranker=ranker,
    movies=movies_df,
)

print(f"\nTop-10 recommendations for user {demo_uid_} (loaded models):")
for rank, r in enumerate(loaded_recs, 1):
    print(f"  {rank:2d}. {r['title']:<45}  score={r['score']:.4f}")

Enter user id: 199999
Cold-start user 199999 → using popularity + sentiment fallback

Top-10 recommendations for user 199999 (loaded models):
   1. Minions                                        score=3.7187
   2. Interstellar                                   score=3.3388
   3. Deadpool                                       score=2.4892
   4. Guardians of the Galaxy                        score=2.4096
   5. Mad Max: Fury Road                             score=2.1343
   6. Jurassic World                                 score=1.9916
   7. Dawn of the Planet of the Apes                 score=1.4648
   8. Pirates of the Caribbean: The Curse of the Black Pearl  score=1.3122
   9. Big Hero 6                                     score=1.2771
  10. The Dark Knight                                score=1.2023
